# 🫁 Multi-Scale Attention-Guided Pneumonia Detection
### AMS 563 — Medical Image Analysis — Final Project

This notebook runs all 6 experiments comparing:
- **E1**: Faster R-CNN baseline
- **E2**: Vanilla DETR baseline
- **E3**: DETR + domain-adaptive pretraining
- **E4**: Multi-Scale DETR (proposed)
- **E5**: DETR + class-aware augmentation
- **E6**: Full pipeline (all components combined)

> ⚠️ **Make sure GPU is enabled**: Runtime → Change runtime type → T4 GPU

## ⚙️ Step 1 — Check GPU & Environment

In [ ]:
import subprocess, torch

print('=== GPU Info ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU detected — please enable GPU in Runtime settings!')

print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Training will be very slow on CPU. Please enable GPU.')

## 📦 Step 2 — Install Dependencies

In [ ]:
!pip install -q pydicom albumentations opencv-python scikit-learn tqdm timm pycocotools scipy
print('✅ All dependencies installed')

## 💾 Step 3 — Mount Google Drive
We save all model checkpoints and results to Drive so they persist after the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/AMS563_Pneumonia'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)
print(f'✅ Drive mounted. Results will be saved to: {DRIVE_DIR}')

## 🔑 Step 4 — Set Up Kaggle API Key
Paste your `kaggle.json` contents below (username + key).

Get it from: https://www.kaggle.com/settings/account → Legacy API Credentials → Create Legacy API Key

In [ ]:
import json, os

# ✏️ FILL IN YOUR KAGGLE CREDENTIALS HERE
kaggle_creds = {
    'username': 'rishabhgosain',   # Your Kaggle username
    'key': 'YOUR_LEGACY_API_KEY_HERE'  # From kaggle.json — the 'key' field
}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Verify
!kaggle competitions list 2>&1 | head -3
print('✅ Kaggle API ready')

## 📥 Step 5 — Download RSNA Dataset (~3.96 GB)

In [ ]:
import os

RAW_DATA = '/content/rsna_data'
os.makedirs(RAW_DATA, exist_ok=True)

# Check if already downloaded (e.g. from Drive cache)
train_img_dir = f'{RAW_DATA}/stage_2_train_images'
if os.path.exists(train_img_dir) and len(os.listdir(train_img_dir)) > 1000:
    print(f'✅ Dataset already present ({len(os.listdir(train_img_dir))} train images)')
else:
    print('Downloading RSNA dataset (~3.96 GB)...')
    !kaggle competitions download -c rsna-pneumonia-detection-challenge -p {RAW_DATA}
    print('Extracting...')
    !cd {RAW_DATA} && unzip -q rsna-pneumonia-detection-challenge.zip && rm rsna-pneumonia-detection-challenge.zip
    print(f'✅ Download complete. Train images: {len(os.listdir(train_img_dir))}')

## 🗂️ Step 6 — Set Up Project Structure

In [ ]:
import os, sys

PROJECT = '/content/pneumonia_detection'
for d in ['configs', 'data', 'models', 'utils', 'scripts', 'outputs']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)

# Add to Python path
sys.path.insert(0, PROJECT)
print(f'✅ Project structure created at {PROJECT}')

### 6a — Write configs/config.py

In [ ]:
%%writefile /content/pneumonia_detection/configs/__init__.py


In [ ]:
%%writefile /content/pneumonia_detection/configs/config.py
import os
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

@dataclass
class DataConfig:
    raw_data_dir: str = '/content/rsna_data'
    processed_dir: str = '/content/rsna_processed'
    image_dir: str = '/content/rsna_processed/images'
    train_csv: str = '/content/rsna_processed/train_split.csv'
    val_csv: str = '/content/rsna_processed/val_split.csv'
    test_csv: str = '/content/rsna_processed/test_split.csv'
    original_csv: str = '/content/rsna_data/stage_2_train_labels.csv'
    unlabeled_dir: str = '/content/nih_chestxray/images'
    image_size: int = 512
    num_workers: int = 2
    pin_memory: bool = True
    train_ratio: float = 0.8
    val_ratio: float = 0.1
    test_ratio: float = 0.1

@dataclass
class AugmentationConfig:
    horizontal_flip_p: float = 0.5
    brightness_limit: float = 0.2
    contrast_limit: float = 0.2
    positive_oversample_factor: int = 2
    rotation_limit: int = 10
    scale_range: Tuple[float, float] = (0.9, 1.1)
    mosaic_p: float = 0.3
    mode: str = 'standard'

@dataclass
class ModelConfig:
    backbone: str = 'resnet50'
    pretrained_backbone: bool = True
    freeze_backbone_bn: bool = True
    num_queries: int = 100
    hidden_dim: int = 256
    nheads: int = 8
    num_encoder_layers: int = 6
    num_decoder_layers: int = 6
    dim_feedforward: int = 2048
    dropout: float = 0.1
    num_feature_levels: int = 4
    num_deformable_points: int = 4
    num_classes: int = 2
    score_threshold: float = 0.5
    nms_iou_threshold: float = 0.5

@dataclass
class TrainConfig:
    optimizer: str = 'adamw'
    lr_backbone: float = 1e-5
    lr_heads: float = 1e-4
    weight_decay: float = 1e-4
    clip_max_norm: float = 0.1
    scheduler: str = 'cosine'
    warmup_epochs: int = 5
    min_lr: float = 1e-7
    epochs: int = 30
    batch_size: int = 8
    accumulation_steps: int = 2
    early_stopping_patience: int = 7
    cost_class: float = 1.0
    cost_bbox: float = 5.0
    cost_giou: float = 2.0
    use_focal_loss: bool = True
    focal_alpha: float = 0.25
    focal_gamma: float = 2.0
    output_dir: str = '/content/drive/MyDrive/AMS563_Pneumonia/outputs'
    save_every: int = 5
    log_every: int = 50
    use_wandb: bool = False
    project_name: str = 'pneumonia_detection'

@dataclass
class MAEConfig:
    mask_ratio: float = 0.75
    decoder_dim: int = 512
    decoder_depth: int = 8
    decoder_heads: int = 16
    epochs: int = 100
    batch_size: int = 32
    lr: float = 1.5e-4
    warmup_epochs: int = 10
    image_size: int = 224

@dataclass
class EvalConfig:
    iou_thresholds: List[float] = field(
        default_factory=lambda: [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
    )
    small_max: int = 96
    medium_max: int = 256
    num_vis_samples: int = 50
    save_attention_maps: bool = True

@dataclass
class Config:
    data: DataConfig = field(default_factory=DataConfig)
    augmentation: AugmentationConfig = field(default_factory=AugmentationConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    mae: MAEConfig = field(default_factory=MAEConfig)
    eval: EvalConfig = field(default_factory=EvalConfig)
    experiment_name: str = 'default'
    seed: int = 42
    device: str = 'cuda'

    def get_output_dir(self):
        path = os.path.join(self.train.output_dir, self.experiment_name)
        os.makedirs(path, exist_ok=True)
        return path

def get_config(**overrides) -> Config:
    cfg = Config()
    for key, value in overrides.items():
        parts = key.split('.')
        obj = cfg
        for p in parts[:-1]:
            obj = getattr(obj, p)
        setattr(obj, parts[-1], value)
    return cfg

### 6b — Write data/dataset.py

In [ ]:
%%writefile /content/pneumonia_detection/data/__init__.py


In [ ]:
%%writefile /content/pneumonia_detection/data/dataset.py
import os
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from configs.config import Config

class RSNAPneumoniaDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform=None, image_size=512, is_train=True):
        self.image_dir = image_dir
        self.transform = transform
        self.image_size = image_size
        self.is_train = is_train
        df = pd.read_csv(csv_path)
        self.patient_ids = df['patientId'].unique().tolist()
        self.annotations = {}
        for pid in self.patient_ids:
            patient_rows = df[df['patientId'] == pid]
            boxes = []
            for _, row in patient_rows.iterrows():
                if row['Target'] == 1 and pd.notna(row.get('x')):
                    x1 = float(row['x'])
                    y1 = float(row['y'])
                    x2 = x1 + float(row['w'])
                    y2 = y1 + float(row['h'])
                    boxes.append([x1, y1, x2, y2])
            self.annotations[pid] = boxes
        self.positive_indices = [i for i, pid in enumerate(self.patient_ids) if len(self.annotations[pid]) > 0]
        self.negative_indices = [i for i, pid in enumerate(self.patient_ids) if len(self.annotations[pid]) == 0]
        print(f'  Loaded {len(self.patient_ids)} images ({len(self.positive_indices)} positive, {len(self.negative_indices)} negative)')

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        img_path = os.path.join(self.image_dir, f'{pid}.png')
        image = Image.open(img_path).convert('RGB')
        boxes = self.annotations[pid]
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.ones(len(boxes), dtype=torch.int64)
            areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
            areas = torch.zeros(0, dtype=torch.float32)
        target = {'boxes': boxes, 'labels': labels, 'image_id': torch.tensor([idx]),
                  'area': areas, 'iscrowd': torch.zeros(len(boxes), dtype=torch.int64), 'patient_id': pid}
        if self.transform is not None:
            image, target = self._apply_transform(image, target)
        else:
            image = self._default_transform(image)
        return image, target

    def _default_transform(self, image):
        image = image.resize((self.image_size, self.image_size))
        image = np.array(image, dtype=np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image = (image - mean) / std
        return torch.from_numpy(image).permute(2, 0, 1).float()

    def _apply_transform(self, image, target):
        image = image.resize((self.image_size, self.image_size))
        image_np = np.array(image)
        boxes = target['boxes'].numpy()
        labels = target['labels'].numpy()
        if len(boxes) > 0:
            transformed = self.transform(image=image_np, bboxes=boxes.tolist(), labels=labels.tolist())
        else:
            transformed = self.transform(image=image_np, bboxes=[], labels=[])
        image_np = transformed['image'].astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image_np = (image_np - mean) / std
        image_tensor = torch.from_numpy(image_np).permute(2, 0, 1).float()
        if len(transformed['bboxes']) > 0:
            target['boxes'] = torch.as_tensor(transformed['bboxes'], dtype=torch.float32)
            target['labels'] = torch.as_tensor(transformed['labels'], dtype=torch.int64)
            target['area'] = (target['boxes'][:, 2] - target['boxes'][:, 0]) * (target['boxes'][:, 3] - target['boxes'][:, 1])
            target['iscrowd'] = torch.zeros(len(target['boxes']), dtype=torch.int64)
        else:
            target['boxes'] = torch.zeros((0, 4), dtype=torch.float32)
            target['labels'] = torch.zeros(0, dtype=torch.int64)
            target['area'] = torch.zeros(0, dtype=torch.float32)
            target['iscrowd'] = torch.zeros(0, dtype=torch.int64)
        return image_tensor, target

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch], dim=0)
    targets = [item[1] for item in batch]
    return images, targets

def build_dataloaders(cfg, train_transform=None, val_transform=None):
    print('Building dataloaders...')
    train_dataset = RSNAPneumoniaDataset(cfg.data.train_csv, cfg.data.image_dir, train_transform, cfg.data.image_size, True)
    val_dataset = RSNAPneumoniaDataset(cfg.data.val_csv, cfg.data.image_dir, val_transform, cfg.data.image_size, False)
    test_dataset = RSNAPneumoniaDataset(cfg.data.test_csv, cfg.data.image_dir, val_transform, cfg.data.image_size, False)
    sampler, shuffle = None, True
    if cfg.augmentation.mode == 'class_aware':
        weights = np.ones(len(train_dataset))
        for i in train_dataset.positive_indices:
            weights[i] = cfg.augmentation.positive_oversample_factor
        sampler = WeightedRandomSampler(weights=weights, num_samples=len(train_dataset), replacement=True)
        shuffle = False
    train_loader = DataLoader(train_dataset, batch_size=cfg.train.batch_size,
        shuffle=shuffle if sampler is None else False, sampler=sampler,
        num_workers=cfg.data.num_workers, pin_memory=cfg.data.pin_memory,
        collate_fn=collate_fn, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg.train.batch_size,
        shuffle=False, num_workers=cfg.data.num_workers,
        pin_memory=cfg.data.pin_memory, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
        num_workers=cfg.data.num_workers, pin_memory=cfg.data.pin_memory, collate_fn=collate_fn)
    return train_loader, val_loader, test_loader

### 6c — Write data/augmentations.py

In [ ]:
%%writefile /content/pneumonia_detection/data/augmentations.py
import albumentations as A
import cv2

def get_standard_augmentation(image_size=512):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=5,
                           border_mode=cv2.BORDER_CONSTANT, value=0, p=0.3),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.3))

def get_class_aware_augmentation(image_size=512):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=10,
                           border_mode=cv2.BORDER_CONSTANT, value=0, p=0.7),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=1.0),
            A.RandomGamma(gamma_limit=(80, 120), p=1.0),
        ], p=0.7),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.GaussNoise(var_limit=(10.0, 30.0), p=1.0),
        ], p=0.3),
        A.Perspective(scale=(0.02, 0.05), p=0.3),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.3))

def get_validation_augmentation(image_size=512):
    return A.Compose([], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.3))

def build_augmentation(mode, image_size=512):
    if mode == 'none': return get_validation_augmentation(image_size)
    elif mode == 'standard': return get_standard_augmentation(image_size)
    elif mode == 'class_aware': return get_class_aware_augmentation(image_size)
    else: raise ValueError(f'Unknown augmentation mode: {mode}')

### 6d — Write all model files

In [ ]:
%%writefile /content/pneumonia_detection/models/__init__.py


In [ ]:
%%writefile /content/pneumonia_detection/models/faster_rcnn.py
import torch, torch.nn as nn, torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
from configs.config import ModelConfig

def build_faster_rcnn(cfg, pretrained=True):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        pretrained=pretrained, min_size=512, max_size=512)
    anchor_sizes = ((32,), (64,), (128,), (256,), (512,))
    anchor_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
    model.rpn.anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=anchor_ratios)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, cfg.num_classes)
    model.roi_heads.score_thresh = cfg.score_threshold
    model.roi_heads.nms_thresh = cfg.nms_iou_threshold
    return model

class FasterRCNNWrapper(nn.Module):
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        self.model = build_faster_rcnn(cfg, pretrained)
    def forward(self, images, targets=None):
        if self.training and targets is not None:
            return self.model([img for img in images], [{'boxes': t['boxes'], 'labels': t['labels']} for t in targets])
        return self.model([img for img in images])

In [ ]:
%%writefile /content/pneumonia_detection/models/detr_baseline.py
import math
from typing import Dict, List
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import resnet50
from torchvision.ops import box_convert
from configs.config import ModelConfig

class PositionalEncoding2D(nn.Module):
    def __init__(self, d_model, temperature=10000.0):
        super().__init__()
        self.d_model = d_model
        self.temperature = temperature
    def forward(self, x):
        B, C, H, W = x.shape
        half_d = self.d_model // 2
        y_embed = torch.arange(H, device=x.device).float().unsqueeze(1).expand(H, W)
        x_embed = torch.arange(W, device=x.device).float().unsqueeze(0).expand(H, W)
        y_embed = y_embed / (H + 1e-6) * 2 * math.pi
        x_embed = x_embed / (W + 1e-6) * 2 * math.pi
        dim_t = torch.arange(half_d // 2, device=x.device).float()
        dim_t = self.temperature ** (2 * dim_t / half_d)
        pos_x = x_embed.unsqueeze(-1) / dim_t
        pos_y = y_embed.unsqueeze(-1) / dim_t
        pos_x = torch.stack([pos_x.sin(), pos_x.cos()], dim=-1).flatten(-2)
        pos_y = torch.stack([pos_y.sin(), pos_y.cos()], dim=-1).flatten(-2)
        pos = torch.cat([pos_x, pos_y], dim=-1).permute(2, 0, 1).unsqueeze(0).expand(B, -1, -1, -1)
        return pos

class DETRBackbone(nn.Module):
    def __init__(self, hidden_dim=256, pretrained=True):
        super().__init__()
        backbone = resnet50(pretrained=pretrained)
        self.body = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
                                   backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4)
        self.proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
    def forward(self, x):
        return self.proj(self.body(x))

class DETR(nn.Module):
    def __init__(self, cfg, pretrained_backbone=True):
        super().__init__()
        self.cfg = cfg
        self.num_classes = cfg.num_classes
        self.num_queries = cfg.num_queries
        self.hidden_dim = cfg.hidden_dim
        self.backbone = DETRBackbone(cfg.hidden_dim, pretrained_backbone)
        self.pos_encoder = PositionalEncoding2D(cfg.hidden_dim)
        enc = nn.TransformerEncoderLayer(cfg.hidden_dim, cfg.nheads, cfg.dim_feedforward, cfg.dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=cfg.num_encoder_layers)
        dec = nn.TransformerDecoderLayer(cfg.hidden_dim, cfg.nheads, cfg.dim_feedforward, cfg.dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec, num_layers=cfg.num_decoder_layers)
        self.query_embed = nn.Embedding(cfg.num_queries, cfg.hidden_dim)
        self.class_head = nn.Linear(cfg.hidden_dim, cfg.num_classes)
        self.bbox_head = nn.Sequential(nn.Linear(cfg.hidden_dim, cfg.hidden_dim), nn.ReLU(),
                                        nn.Linear(cfg.hidden_dim, cfg.hidden_dim), nn.ReLU(),
                                        nn.Linear(cfg.hidden_dim, 4))
        self._reset_parameters()
    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)
    def forward(self, images):
        B = images.shape[0]
        features = self.backbone(images)
        pos = self.pos_encoder(features)
        src = features.flatten(2).permute(0, 2, 1)
        pos_flat = pos.flatten(2).permute(0, 2, 1)
        memory = self.encoder(src + pos_flat)
        query = self.query_embed.weight.unsqueeze(0).expand(B, -1, -1)
        hs = self.decoder(query, memory)
        return {'pred_logits': self.class_head(hs), 'pred_boxes': self.bbox_head(hs).sigmoid()}
    @torch.no_grad()
    def predict(self, images, score_threshold=0.5):
        outputs = self.forward(images)
        results = []
        for i in range(images.shape[0]):
            probs = outputs['pred_logits'][i].softmax(-1)
            scores = probs[:, 1]
            keep = scores > score_threshold
            boxes_xyxy = box_convert(outputs['pred_boxes'][i][keep], 'cxcywh', 'xyxy')
            boxes_xyxy[:, [0, 2]] *= images.shape[3]
            boxes_xyxy[:, [1, 3]] *= images.shape[2]
            results.append({'boxes': boxes_xyxy, 'scores': scores[keep],
                            'labels': torch.ones(keep.sum(), dtype=torch.long)})
        return results

def build_detr(cfg, pretrained=True):
    return DETR(cfg, pretrained_backbone=pretrained)

In [ ]:
%%writefile /content/pneumonia_detection/models/detr_multiscale.py
import math
from typing import Dict, List
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision.models import resnet50
from torchvision.ops import box_convert
from configs.config import ModelConfig

class FPNBackbone(nn.Module):
    def __init__(self, hidden_dim=256, pretrained=True):
        super().__init__()
        backbone = resnet50(pretrained=pretrained)
        self.layer0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.lateral4 = nn.Conv2d(2048, hidden_dim, 1)
        self.lateral3 = nn.Conv2d(1024, hidden_dim, 1)
        self.lateral2 = nn.Conv2d(512, hidden_dim, 1)
        self.output4 = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1)
        self.output3 = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1)
        self.output2 = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1)
        self.level_embed = nn.Parameter(torch.randn(3, hidden_dim))
    def forward(self, x):
        x = self.layer0(x)
        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        p5 = self.lateral4(c5)
        p4 = self.lateral3(c4) + F.interpolate(p5, size=c4.shape[2:], mode='nearest')
        p3 = self.lateral2(c3) + F.interpolate(p4, size=c3.shape[2:], mode='nearest')
        p5 = self.output4(p5) + self.level_embed[2].view(1, -1, 1, 1)
        p4 = self.output3(p4) + self.level_embed[1].view(1, -1, 1, 1)
        p3 = self.output2(p3) + self.level_embed[0].view(1, -1, 1, 1)
        return [p3, p4, p5]

class MultiScalePositionalEncoding(nn.Module):
    def __init__(self, hidden_dim, max_size=128):
        super().__init__()
        self.row_embed = nn.Embedding(max_size, hidden_dim // 2)
        self.col_embed = nn.Embedding(max_size, hidden_dim // 2)
        nn.init.uniform_(self.row_embed.weight)
        nn.init.uniform_(self.col_embed.weight)
    def forward(self, feature):
        H, W = feature.shape[2], feature.shape[3]
        rows = self.row_embed(torch.arange(H, device=feature.device))
        cols = self.col_embed(torch.arange(W, device=feature.device))
        pos = torch.cat([cols.unsqueeze(0).expand(H, -1, -1), rows.unsqueeze(1).expand(-1, W, -1)], dim=-1)
        return pos.permute(2, 0, 1).unsqueeze(0).expand(feature.shape[0], -1, -1, -1)

class MultiScaleDETR(nn.Module):
    def __init__(self, cfg, pretrained_backbone=True):
        super().__init__()
        self.cfg = cfg
        self.num_classes = cfg.num_classes
        self.num_queries = cfg.num_queries
        self.hidden_dim = cfg.hidden_dim
        self.backbone = FPNBackbone(cfg.hidden_dim, pretrained_backbone)
        self.pos_encoders = nn.ModuleList([MultiScalePositionalEncoding(cfg.hidden_dim) for _ in range(3)])
        enc = nn.TransformerEncoderLayer(cfg.hidden_dim, cfg.nheads, cfg.dim_feedforward, cfg.dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=cfg.num_encoder_layers)
        dec = nn.TransformerDecoderLayer(cfg.hidden_dim, cfg.nheads, cfg.dim_feedforward, cfg.dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec, num_layers=cfg.num_decoder_layers)
        self.query_embed = nn.Embedding(cfg.num_queries, cfg.hidden_dim)
        self.class_head = nn.Linear(cfg.hidden_dim, cfg.num_classes)
        self.bbox_head = nn.Sequential(nn.Linear(cfg.hidden_dim, cfg.hidden_dim), nn.ReLU(),
                                        nn.Linear(cfg.hidden_dim, cfg.hidden_dim), nn.ReLU(),
                                        nn.Linear(cfg.hidden_dim, 4))
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)
    def forward(self, images):
        B = images.shape[0]
        feats = self.backbone(images)
        flat = []
        for i, (feat, enc) in enumerate(zip(feats, self.pos_encoders)):
            feat = feat + enc(feat)
            flat.append(feat.flatten(2).permute(0, 2, 1))
        tokens = torch.cat(flat, dim=1)
        memory = self.encoder(tokens)
        query = self.query_embed.weight.unsqueeze(0).expand(B, -1, -1)
        hs = self.decoder(query, memory)
        return {'pred_logits': self.class_head(hs), 'pred_boxes': self.bbox_head(hs).sigmoid()}
    @torch.no_grad()
    def predict(self, images, score_threshold=0.5):
        outputs = self.forward(images)
        results = []
        for i in range(images.shape[0]):
            probs = outputs['pred_logits'][i].softmax(-1)
            scores = probs[:, 1]
            keep = scores > score_threshold
            boxes_xyxy = box_convert(outputs['pred_boxes'][i][keep], 'cxcywh', 'xyxy')
            boxes_xyxy[:, [0, 2]] *= images.shape[3]
            boxes_xyxy[:, [1, 3]] *= images.shape[2]
            results.append({'boxes': boxes_xyxy, 'scores': scores[keep],
                            'labels': torch.ones(keep.sum(), dtype=torch.long)})
        return results

def build_detr_multiscale(cfg, pretrained=True):
    return MultiScaleDETR(cfg, pretrained_backbone=pretrained)

### 6e — Write utils files

In [ ]:
%%writefile /content/pneumonia_detection/utils/__init__.py


In [ ]:
# Copy losses, metrics, engine from the project — these are large files
# We'll write condensed but complete versions
import shutil, os

# Write the losses module
losses_code = '''
from typing import Dict, List, Tuple
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.optimize import linear_sum_assignment
from torchvision.ops import generalized_box_iou, box_convert

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction="none")
        p_t = torch.exp(-ce_loss)
        return (self.alpha * (1 - p_t) ** self.gamma * ce_loss).mean()

class HungarianMatcher(nn.Module):
    def __init__(self, cost_class=1.0, cost_bbox=5.0, cost_giou=2.0):
        super().__init__()
        self.cost_class = cost_class
        self.cost_bbox = cost_bbox
        self.cost_giou = cost_giou
    @torch.no_grad()
    def forward(self, outputs, targets):
        B, Q, C = outputs["pred_logits"].shape
        pred_logits = outputs["pred_logits"].flatten(0, 1).softmax(-1)
        pred_boxes = outputs["pred_boxes"].flatten(0, 1)
        tgt_labels = torch.cat([t["labels"] for t in targets])
        tgt_boxes = torch.cat([t["boxes"] for t in targets])
        if len(tgt_labels) == 0:
            return [(torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)) for _ in range(B)]
        cost_class = -pred_logits[:, tgt_labels]
        cost_bbox = torch.cdist(pred_boxes, tgt_boxes, p=1)
        pred_boxes_xyxy = box_convert(pred_boxes, "cxcywh", "xyxy")
        tgt_boxes_xyxy = tgt_boxes if tgt_boxes.shape[-1] == 4 else box_convert(tgt_boxes, "cxcywh", "xyxy")
        try:
            cost_giou = -generalized_box_iou(pred_boxes_xyxy, tgt_boxes_xyxy)
        except:
            cost_giou = torch.zeros_like(cost_bbox)
        cost = (self.cost_class * cost_class + self.cost_bbox * cost_bbox + self.cost_giou * cost_giou)
        cost = cost.view(B, Q, -1).cpu()
        sizes = [len(t["labels"]) for t in targets]
        indices, offset = [], 0
        for i, s in enumerate(sizes):
            if s == 0:
                indices.append((torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)))
            else:
                pi, ti = linear_sum_assignment(cost[i, :, offset:offset+s].numpy())
                indices.append((torch.as_tensor(pi, dtype=torch.long), torch.as_tensor(ti, dtype=torch.long)))
            offset += s
        return indices

class DETRLoss(nn.Module):
    def __init__(self, num_classes=2, cost_class=1.0, cost_bbox=5.0, cost_giou=2.0,
                 use_focal=True, focal_alpha=0.25, focal_gamma=2.0, eos_coef=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.matcher = HungarianMatcher(cost_class, cost_bbox, cost_giou)
        self.cost_class = cost_class
        self.cost_bbox = cost_bbox
        self.cost_giou = cost_giou
        self.use_focal = use_focal
        self.cls_loss_fn = FocalLoss(focal_alpha, focal_gamma) if use_focal else None
        if not use_focal:
            w = torch.ones(num_classes)
            w[-1] = eos_coef
            self.register_buffer("class_weight", w)
    def forward(self, outputs, targets):
        IMAGE_SIZE = 512
        normalized_targets = []
        for t in targets:
            nt = {k: v for k, v in t.items()}
            if len(t["boxes"]) > 0:
                boxes = box_convert(t["boxes"].clone(), "xyxy", "cxcywh")
                boxes[:, [0, 2]] /= IMAGE_SIZE
                boxes[:, [1, 3]] /= IMAGE_SIZE
                nt["boxes"] = boxes.clamp(0, 1)
            normalized_targets.append(nt)
        indices = self.matcher(outputs, normalized_targets)
        B, Q, C = outputs["pred_logits"].shape
        device = outputs["pred_logits"].device
        target_classes = torch.zeros(B, Q, dtype=torch.long, device=device)
        for i, (pi, ti) in enumerate(indices):
            if len(pi) > 0:
                target_classes[i, pi] = normalized_targets[i]["labels"][ti].to(device)
        if self.use_focal:
            loss_ce = self.cls_loss_fn(outputs["pred_logits"].reshape(-1, C), target_classes.reshape(-1))
        else:
            loss_ce = F.cross_entropy(outputs["pred_logits"].reshape(-1, C), target_classes.reshape(-1), weight=self.class_weight.to(device))
        loss_bbox = torch.tensor(0.0, device=device)
        loss_giou = torch.tensor(0.0, device=device)
        n_boxes = 0
        for i, (pi, ti) in enumerate(indices):
            if len(pi) == 0: continue
            pb = outputs["pred_boxes"][i, pi]
            tb = normalized_targets[i]["boxes"][ti].to(device)
            loss_bbox = loss_bbox + F.l1_loss(pb, tb, reduction="sum")
            giou = generalized_box_iou(box_convert(pb, "cxcywh", "xyxy"), box_convert(tb, "cxcywh", "xyxy"))
            loss_giou = loss_giou + (1 - giou.diag()).sum()
            n_boxes += len(pi)
        if n_boxes > 0:
            loss_bbox = loss_bbox / n_boxes
            loss_giou = loss_giou / n_boxes
        total = self.cost_class * loss_ce + self.cost_bbox * loss_bbox + self.cost_giou * loss_giou
        return {"loss_ce": loss_ce, "loss_bbox": loss_bbox, "loss_giou": loss_giou, "total_loss": total}
'''
with open('/content/pneumonia_detection/utils/losses.py', 'w') as f:
    f.write(losses_code)
print('✅ utils/losses.py written')

In [ ]:
%%writefile /content/pneumonia_detection/utils/metrics.py
from typing import Dict, List, Tuple
import numpy as np
import torch
from torchvision.ops import box_iou

def compute_iou_matrix(pred_boxes, gt_boxes):
    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return torch.zeros(len(pred_boxes), len(gt_boxes))
    return box_iou(pred_boxes, gt_boxes)

def compute_ap(recalls, precisions):
    ap = 0.0
    for t in np.arange(0.0, 1.1, 0.1):
        p = np.max(precisions[recalls >= t]) if np.sum(recalls >= t) > 0 else 0
        ap += p / 11.0
    return ap

def _compute_ap_at_threshold(predictions, targets, iou_threshold, min_area=0, max_area=float('inf')):
    all_scores, all_tp, n_gt = [], [], 0
    for pred, tgt in zip(predictions, targets):
        pred_boxes = pred['boxes']
        pred_scores = pred['scores']
        gt_boxes = tgt['boxes']
        if len(gt_boxes) > 0:
            gt_areas = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])
            size_mask = (gt_areas >= min_area) & (gt_areas < max_area)
            gt_boxes = gt_boxes[size_mask]
        n_gt += len(gt_boxes)
        if len(pred_boxes) == 0: continue
        sort_idx = pred_scores.argsort(descending=True)
        pred_boxes = pred_boxes[sort_idx]
        pred_scores = pred_scores[sort_idx]
        if len(gt_boxes) == 0:
            all_scores.extend(pred_scores.cpu().numpy().tolist())
            all_tp.extend([0] * len(pred_scores))
            continue
        iou_matrix = compute_iou_matrix(pred_boxes, gt_boxes)
        matched_gt = set()
        for i in range(len(pred_boxes)):
            all_scores.append(pred_scores[i].item())
            if iou_matrix.shape[1] > 0:
                max_iou, max_idx = iou_matrix[i].max(dim=0)
                if max_iou.item() >= iou_threshold and max_idx.item() not in matched_gt:
                    all_tp.append(1)
                    matched_gt.add(max_idx.item())
                else:
                    all_tp.append(0)
            else:
                all_tp.append(0)
    if n_gt == 0: return 0.0
    sorted_idx = np.argsort(-np.array(all_scores))
    tp = np.array(all_tp)[sorted_idx]
    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(1 - tp)
    recalls = tp_cum / n_gt
    precisions = tp_cum / (tp_cum + fp_cum)
    return compute_ap(recalls, precisions)

def _compute_precision_recall(predictions, targets, iou_threshold=0.5, score_threshold=0.5):
    tp, fp, fn = 0, 0, 0
    for pred, tgt in zip(predictions, targets):
        pred_boxes = pred['boxes'][pred['scores'] >= score_threshold]
        gt_boxes = tgt['boxes']
        if len(gt_boxes) == 0: fp += len(pred_boxes); continue
        if len(pred_boxes) == 0: fn += len(gt_boxes); continue
        iou_matrix = compute_iou_matrix(pred_boxes, gt_boxes)
        matched_gt = set()
        for i in range(len(pred_boxes)):
            max_iou, max_idx = iou_matrix[i].max(dim=0)
            if max_iou.item() >= iou_threshold and max_idx.item() not in matched_gt:
                tp += 1; matched_gt.add(max_idx.item())
            else:
                fp += 1
        fn += len(gt_boxes) - len(matched_gt)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def evaluate_detections(all_predictions, all_targets, iou_thresholds=None, size_ranges=None):
    if iou_thresholds is None: iou_thresholds = [0.5 + 0.05 * i for i in range(10)]
    if size_ranges is None:
        size_ranges = {'small': (0, 96**2), 'medium': (96**2, 256**2), 'large': (256**2, float('inf'))}
    results = {}
    ap_per_threshold = []
    for iou_thr in iou_thresholds:
        ap = _compute_ap_at_threshold(all_predictions, all_targets, iou_thr)
        ap_per_threshold.append(ap)
        results[f'AP@{iou_thr:.2f}'] = ap
    results['mAP@0.5'] = results.get('AP@0.50', 0.0)
    results['mAP@0.5:0.95'] = np.mean(ap_per_threshold)
    for size_name, (min_area, max_area) in size_ranges.items():
        results[f'AP@0.5_{size_name}'] = _compute_ap_at_threshold(all_predictions, all_targets, 0.5, min_area, max_area)
    precision, recall, f1 = _compute_precision_recall(all_predictions, all_targets)
    results['precision@0.5'] = precision
    results['recall@0.5'] = recall
    results['f1@0.5'] = f1
    return results

In [ ]:
%%writefile /content/pneumonia_detection/utils/engine.py
import os, torch, torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from configs.config import Config
from utils.metrics import evaluate_detections

class Trainer:
    def __init__(self, model, criterion, optimizer, scheduler, cfg, train_loader, val_loader, device='cuda'):
        self.model = model.to(device)
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.cfg = cfg
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.output_dir = cfg.get_output_dir()
        self.best_metric = 0.0
        self.patience_counter = 0

    def train(self):
        print(f'\n{"="*60}\nStarting: {self.cfg.experiment_name}\n{"="*60}')
        history = {'train_loss': [], 'val_metrics': []}
        for epoch in range(self.cfg.train.epochs):
            train_loss = self._train_one_epoch(epoch)
            history['train_loss'].append(train_loss)
            val_metrics = self._validate(epoch)
            history['val_metrics'].append(val_metrics)
            if self.scheduler: self.scheduler.step()
            current_metric = val_metrics.get('mAP@0.5', 0.0)
            if current_metric > self.best_metric:
                self.best_metric = current_metric
                self._save_checkpoint(epoch, is_best=True)
                self.patience_counter = 0
                print(f'  >> New best mAP@0.5: {self.best_metric:.4f}')
            else:
                self.patience_counter += 1
            if epoch % self.cfg.train.save_every == 0:
                self._save_checkpoint(epoch, is_best=False)
            if self.patience_counter >= self.cfg.train.early_stopping_patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
        print(f'Done. Best mAP@0.5: {self.best_metric:.4f}')
        return history

    def _train_one_epoch(self, epoch):
        self.model.train()
        total_loss, n_batches = 0, 0
        accum_steps = self.cfg.train.accumulation_steps
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch+1}/{self.cfg.train.epochs}')
        self.optimizer.zero_grad()
        for batch_idx, (images, targets) in enumerate(pbar):
            images = images.to(self.device)
            targets = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
            is_faster_rcnn = hasattr(self.model, 'model') and hasattr(getattr(self.model, 'model', None), 'rpn')
            if is_faster_rcnn:
                loss_dict = self.model(images, targets)
                loss = sum(loss_dict.values())
            else:
                outputs = self.model(images)
                loss_dict = self.criterion(outputs, targets)
                loss = loss_dict['total_loss']
            (loss / accum_steps).backward()
            if (batch_idx + 1) % accum_steps == 0:
                if self.cfg.train.clip_max_norm > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.train.clip_max_norm)
                self.optimizer.step()
                self.optimizer.zero_grad()
            total_loss += loss.item()
            n_batches += 1
            if batch_idx % self.cfg.train.log_every == 0:
                pbar.set_postfix({'loss': f'{total_loss/n_batches:.4f}'})
        return total_loss / max(n_batches, 1)

    @torch.no_grad()
    def _validate(self, epoch):
        self.model.eval()
        all_predictions, all_targets = [], []
        for images, targets in tqdm(self.val_loader, desc='Validating'):
            images = images.to(self.device)
            is_faster_rcnn = hasattr(self.model, 'model') and hasattr(getattr(self.model, 'model', None), 'rpn')
            if is_faster_rcnn:
                predictions = self.model(images)
            else:
                predictions = self.model.predict(images, score_threshold=self.cfg.model.score_threshold)
            for pred in predictions:
                all_predictions.append({k: v.cpu() if isinstance(v, torch.Tensor) else v for k, v in pred.items()})
            for tgt in targets:
                all_targets.append({k: v.cpu() if isinstance(v, torch.Tensor) else v for k, v in tgt.items()})
        metrics = evaluate_detections(all_predictions, all_targets, iou_thresholds=self.cfg.eval.iou_thresholds)
        print(f'  Val [epoch {epoch+1}]: mAP@0.5={metrics["mAP@0.5"]:.4f}  '
              f'mAP@0.5:0.95={metrics["mAP@0.5:0.95"]:.4f}  '
              f'F1={metrics["f1@0.5"]:.4f}')
        return metrics

    def _save_checkpoint(self, epoch, is_best=False):
        checkpoint = {'epoch': epoch, 'model_state_dict': self.model.state_dict(),
                      'optimizer_state_dict': self.optimizer.state_dict(), 'best_metric': self.best_metric}
        path = os.path.join(self.output_dir, 'best_model.pth' if is_best else f'checkpoint_epoch_{epoch}.pth')
        torch.save(checkpoint, path)

## 🔄 Step 7 — Preprocess DICOM Files → PNG

In [ ]:
import os, sys, numpy as np, pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

RAW_DATA  = '/content/rsna_data'
PROCESSED = '/content/rsna_processed'
IMAGE_DIR = f'{PROCESSED}/images'
IMAGE_SIZE = 512

os.makedirs(IMAGE_DIR, exist_ok=True)

def convert_dicom_to_png(dicom_path, output_path, target_size=512):
    try:
        import pydicom, cv2
        ds = pydicom.dcmread(dicom_path)
        pixel_array = ds.pixel_array.astype(np.float32)
        if pixel_array.max() > pixel_array.min():
            pixel_array = (pixel_array - pixel_array.min()) / (pixel_array.max() - pixel_array.min())
        pixel_array = (pixel_array * 255).astype(np.uint8)
        pixel_array = cv2.equalizeHist(pixel_array)
        h, w = pixel_array.shape
        scale = target_size / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        resized = cv2.resize(pixel_array, (new_w, new_h), interpolation=cv2.INTER_AREA)
        canvas = np.zeros((target_size, target_size), dtype=np.uint8)
        y_off = (target_size - new_h) // 2
        x_off = (target_size - new_w) // 2
        canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
        Image.fromarray(canvas).save(output_path)
        return True
    except Exception as e:
        print(f'Error: {dicom_path}: {e}')
        return False

# Check if already done
existing = len(os.listdir(IMAGE_DIR))
if existing > 25000:
    print(f'✅ Already converted {existing} images, skipping.')
else:
    dicom_dir = f'{RAW_DATA}/stage_2_train_images'
    dcm_files = [os.path.join(dicom_dir, f) for f in os.listdir(dicom_dir) if f.endswith('.dcm')]
    print(f'Converting {len(dcm_files)} DICOM files to {IMAGE_SIZE}x{IMAGE_SIZE} PNG...')
    success = 0
    for dcm_path in tqdm(dcm_files):
        pid = os.path.splitext(os.path.basename(dcm_path))[0]
        out_path = f'{IMAGE_DIR}/{pid}.png'
        if not os.path.exists(out_path):
            if convert_dicom_to_png(dcm_path, out_path, IMAGE_SIZE): success += 1
        else:
            success += 1
    print(f'✅ Converted {success}/{len(dcm_files)} files')

# Build train/val/test splits
csv_path = f'{RAW_DATA}/stage_2_train_labels.csv'
df = pd.read_csv(csv_path)
patients = df.groupby('patientId').agg({'Target': 'max'}).reset_index()
train_ids, temp_ids = train_test_split(patients['patientId'].values, test_size=0.2, random_state=42, stratify=patients['Target'].values)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42,
    stratify=patients[patients['patientId'].isin(temp_ids)]['Target'].values)

records = []
for _, row in df.iterrows():
    pid, target = row['patientId'], int(row['Target'])
    rec = {'patientId': pid, 'Target': target, 'image_path': f'images/{pid}.png'}
    if target == 1 and pd.notna(row.get('x')):
        rec.update({'x': float(row['x']), 'y': float(row['y']),
                    'w': float(row['width']), 'h': float(row['height'])})
    else:
        rec.update({'x': None, 'y': None, 'w': None, 'h': None})
    records.append(rec)

all_df = pd.DataFrame(records)
for split_name, split_ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    split_df = all_df[all_df['patientId'].isin(split_ids)]
    split_df.to_csv(f'{PROCESSED}/{split_name}_split.csv', index=False)
    n_pos = split_df[split_df['Target'] == 1]['patientId'].nunique()
    n_total = split_df['patientId'].nunique()
    print(f'  {split_name}: {n_total} patients, {n_pos} positive ({100*n_pos/n_total:.1f}%)')

print('\n✅ Preprocessing complete!')

## 🚀 Step 8 — Training Helper Function
This function trains any model configuration and saves results to Google Drive.

In [ ]:
import sys, os, random, json, torch, numpy as np
sys.path.insert(0, '/content/pneumonia_detection')

from configs.config import get_config
from data.dataset import build_dataloaders
from data.augmentations import build_augmentation
from utils.engine import Trainer
from utils.losses import DETRLoss

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def run_experiment(model_name, experiment_name, augment='standard', epochs=30, batch_size=8):
    print(f'\n{"="*60}')
    print(f'EXPERIMENT: {experiment_name}')
    print(f'  Model: {model_name} | Augment: {augment} | Epochs: {epochs}')
    print(f'{"="*60}')

    set_seed(42)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'  Device: {device}')

    # Config
    cfg = get_config(
        experiment_name=experiment_name,
        **{'augmentation.mode': augment, 'train.epochs': epochs, 'train.batch_size': batch_size}
    )

    # Dataloaders
    train_transform = build_augmentation(augment, cfg.data.image_size)
    val_transform   = build_augmentation('none',  cfg.data.image_size)
    train_loader, val_loader, test_loader = build_dataloaders(cfg, train_transform, val_transform)

    # Model
    if model_name == 'faster_rcnn':
        from models.faster_rcnn import FasterRCNNWrapper
        model = FasterRCNNWrapper(cfg.model, pretrained=True)
        criterion = None
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=cfg.train.lr_heads, weight_decay=cfg.train.weight_decay)
    elif model_name == 'detr':
        from models.detr_baseline import build_detr
        model = build_detr(cfg.model, pretrained=True)
        criterion = DETRLoss(cfg.model.num_classes, cfg.train.cost_class, cfg.train.cost_bbox,
                             cfg.train.cost_giou, cfg.train.use_focal_loss, cfg.train.focal_alpha, cfg.train.focal_gamma)
        backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' in n]
        head_params     = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' not in n]
        optimizer = torch.optim.AdamW([{'params': backbone_params, 'lr': cfg.train.lr_backbone},
                                       {'params': head_params, 'lr': cfg.train.lr_heads}],
                                      weight_decay=cfg.train.weight_decay)
    elif model_name == 'detr_multiscale':
        from models.detr_multiscale import build_detr_multiscale
        model = build_detr_multiscale(cfg.model, pretrained=True)
        criterion = DETRLoss(cfg.model.num_classes, cfg.train.cost_class, cfg.train.cost_bbox,
                             cfg.train.cost_giou, cfg.train.use_focal_loss, cfg.train.focal_alpha, cfg.train.focal_gamma)
        backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' in n]
        head_params     = [p for n, p in model.named_parameters() if p.requires_grad and 'backbone' not in n]
        optimizer = torch.optim.AdamW([{'params': backbone_params, 'lr': cfg.train.lr_backbone},
                                       {'params': head_params, 'lr': cfg.train.lr_heads}],
                                      weight_decay=cfg.train.weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.train.epochs - cfg.train.warmup_epochs, eta_min=cfg.train.min_lr)

    trainer = Trainer(model=model, criterion=criterion, optimizer=optimizer, scheduler=scheduler,
                      cfg=cfg, train_loader=train_loader, val_loader=val_loader, device=device)
    history = trainer.train()

    # Save history to Drive
    output_dir = cfg.get_output_dir()
    with open(f'{output_dir}/history.json', 'w') as f:
        json.dump({'model': model_name, 'augment': augment, 'experiment': experiment_name,
                   'train_loss': history['train_loss'],
                   'val_metrics': [{k: float(v) for k, v in m.items()} for m in history['val_metrics']]}, f, indent=2)

    best_map = max(m.get('mAP@0.5', 0) for m in history['val_metrics'])
    print(f'\n✅ {experiment_name} complete. Best mAP@0.5: {best_map:.4f}')
    print(f'   Results saved to: {output_dir}')
    return history, best_map

print('✅ Training helper ready!')

## 🧪 Step 9 — Run All Experiments
Each cell below runs one experiment. Run them **sequentially** — each takes ~30–90 min on a T4 GPU.

Results are automatically saved to your Google Drive after each experiment.

In [ ]:
# E1: Faster R-CNN Baseline
history_e1, map_e1 = run_experiment('faster_rcnn', 'E1_faster_rcnn', augment='standard', epochs=30)

In [ ]:
# E2: Vanilla DETR Baseline
history_e2, map_e2 = run_experiment('detr', 'E2_detr_vanilla', augment='standard', epochs=30)

In [ ]:
# E3: DETR + Class-Aware Augmentation
history_e3, map_e3 = run_experiment('detr', 'E3_detr_augmented', augment='class_aware', epochs=30)

In [ ]:
# E4: Multi-Scale DETR (Proposed Model)
history_e4, map_e4 = run_experiment('detr_multiscale', 'E4_detr_multiscale', augment='standard', epochs=30)

In [ ]:
# E5: Multi-Scale DETR + Class-Aware Augmentation
history_e5, map_e5 = run_experiment('detr_multiscale', 'E5_multiscale_augmented', augment='class_aware', epochs=30)

In [ ]:
# E6: Full Pipeline (Multi-Scale + Class-Aware Aug)
history_e6, map_e6 = run_experiment('detr_multiscale', 'E6_full_pipeline', augment='class_aware', epochs=40)

## 📊 Step 10 — Results Comparison Table & Plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0d1117'
matplotlib.rcParams['axes.facecolor'] = '#161b22'
matplotlib.rcParams['text.color'] = 'white'
matplotlib.rcParams['axes.labelcolor'] = 'white'
matplotlib.rcParams['xtick.color'] = 'white'
matplotlib.rcParams['ytick.color'] = 'white'

# Collect results
experiment_results = {
    'E1 Faster R-CNN': {'map': map_e1, 'history': history_e1, 'color': '#58a6ff'},
    'E2 DETR Vanilla': {'map': map_e2, 'history': history_e2, 'color': '#f78166'},
    'E3 DETR + Aug':   {'map': map_e3, 'history': history_e3, 'color': '#3fb950'},
    'E4 MultiScale':   {'map': map_e4, 'history': history_e4, 'color': '#d2a8ff'},
    'E5 MS + Aug':     {'map': map_e5, 'history': history_e5, 'color': '#ffa657'},
    'E6 Full Pipeline':{'map': map_e6, 'history': history_e6, 'color': '#ff7b72'},
}

print('\n' + '='*55)
print(f'{"Experiment":<22} {"mAP@0.5":>10}')
print('='*55)
for name, res in experiment_results.items():
    print(f'{name:<22} {res["map"]:>10.4f}')
print('='*55)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Pneumonia Detection — Experiment Results', color='white', fontsize=14, fontweight='bold')

# Bar chart
ax1 = axes[0]
names = list(experiment_results.keys())
maps  = [experiment_results[n]['map'] for n in names]
colors = [experiment_results[n]['color'] for n in names]
bars = ax1.bar(names, maps, color=colors, alpha=0.85, edgecolor='#30363d', linewidth=1.2)
for bar, val in zip(bars, maps):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{val:.3f}',
             ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')
ax1.set_title('mAP@0.5 by Experiment', color='white')
ax1.set_ylabel('mAP@0.5')
ax1.set_ylim(0, max(maps) * 1.2 + 0.01)
ax1.tick_params(axis='x', rotation=30)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Training curves
ax2 = axes[1]
for name, res in experiment_results.items():
    val_maps = [m.get('mAP@0.5', 0) for m in res['history']['val_metrics']]
    ax2.plot(val_maps, label=name, color=res['color'], linewidth=2, marker='o', markersize=3)
ax2.set_title('Validation mAP@0.5 Over Epochs', color='white')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('mAP@0.5')
ax2.legend(fontsize=8, facecolor='#161b22', labelcolor='white')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plot_path = '/content/drive/MyDrive/AMS563_Pneumonia/results_comparison.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'\n✅ Plot saved to: {plot_path}')